In [ ]:
pip install -U langchain pypdf pymupdf sentence-transformers chromadb

In [ ]:
form langchain_core.documents import Document

In [ ]:
from langchain_core.documents import Document

In [ ]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {"source": "https://www.google.com"}
)

In [ ]:
sample_doc

## Text Data

In [ ]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader('/content/drive/MyDrive/GenAI/RAG/python.txt', encoding = 'utf-8')
document = loader.load()

In [ ]:
document

## PDF Data

In [ ]:
# pdf_data
from langchain_community.document_loaders import PyMuPDFLoader
pdf_loader = PyMuPDFLoader("/content/drive/MyDrive/GenAI/RAG/Research2.pdf")
document = pdf_loader.load()
document

## Ingestion Pipeline

In [ ]:
# Data => Documents
import os
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
def load_pdfs():
  folder_path = "/content/drive/MyDrive/GenAI/RAG"
  num_docs = 0
  all_docs = []

  for filename in os.listdir(folder_path):
    if filename.endswith(".pdf"):
      # Complete file path
      pdf_path = os.path.join(folder_path, filename)
      
      loader = PyPDFLoader(pdf_path)
      doc = loader.load()

      all_docs.extend(doc)
      num_docs += 1

  print("Total pdfs:", num_docs)
  print("total pages:", len(all_docs))
  return all_docs

In [ ]:
all_pdf_documents = load_pdfs()

## Chucks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_docs(documents, size=500, overlap=50 ):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size= size,
      chunk_overlap= overlap)
  
  chunked_doc = text_splitter.split_documents(documents)
  return chunked_doc

 ## Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbeddingManager:
  def __init__(self, model_name = "all-MiniLM-L6-v2"):
    self.model_name = model_name
    print("Loading model.....", self.model_name)

    self.model = SentenceTransformer(self.model_name)
    print("Embedding dimension=", self.model.get_sentence_embedding_dimension())

  def generate_embeddings(self, text):
    embedding = self.model.encode(text, show_progress_bar=True)
    print("Embedding shape=", embedding.shape)
    return embedding

In [ ]:
embedding_manager = EmbeddingManager()

## Vector Store

In [ ]:
class VectorStoreManager:
  def __init__(self, persist_directory = "/content/drive/MyDrive/GenAI/RAG/VectorStore", collection_name = "my_collection"):
    self.persist_directory = persist_directory
    self.collection_name = collection_name
    self.client = None
    self.collection = None

    self._initialized_store()

  def _initialized_store(self):
    os.makedirs(self.persist_directory, exist_ok = True)

    self.client = chromadb.PersistentClient(path = self.persist_directory)
    self.collection = self.client.get_or_create_collection(
        name = self.collection_name,
        metadata = {"description": "vector store collection for pdf embeddings in RAG"}
    )

    print("Initialized the vector store with collection :", self.collection_name)
    print("docs in collection", self.collection.count())

  def add_documnets(self, documents, embeddings):
    if len(documents)!= len(embeddings):
      raise ValueError("num of documents and embeddings must be same")

    # Store => ids, embeddings, documents, metadatas
    ids = []
    all_metadata = []
    documents_content = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)
      
      metadata = dict(doc.metadata)
      metadata["doc_index"] = i
      metadata["content_length"] = len(doc.page_content)
      all_metadata.append(metadata)

      documents_content.append(doc.page_content)

      embeddings_list.append(embedding.tolist())

      self.collection.add(
          ids = ids,
          embeddings = embeddings_list,
          documents = documents_content,
          metadatas = all_metadata
      )

    print("total documents added in vector store = ", len(documents_content))
    print("docs in collection:", self.collection.count())


In [ ]:
vector_store_manager = VectorStoreManager()

## Retrieval Pipeline

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class RAGRetriver:
  def __init__(self, embedding_manager, vector_store ):
    self.embedding_manager = embedding_manager
    self.vector_store = vector_store